# Tool Use（ツール使用）

Claudeに外部機能（ツール）を提供して、リアルタイム情報の取得やアクション実行を可能にします。

**今回作る3つのツール：**
1. `get_current_datetime` → 現在の日時を取得
2. `add_time_to_datetime` → 日付に期間を加算
3. `set_reminder` → リマインダーを設定

## 1. セットアップ

In [59]:
from dotenv import load_dotenv
load_dotenv()

from anthropic import Anthropic
from datetime import datetime, timedelta

client = Anthropic()
model = "claude-sonnet-4-5"

print("セットアップ完了")

セットアップ完了


## 2. ツール関数の実装

ベストプラクティス：
- 分かりやすい関数名・パラメータ名にする
- 入力値を検証する
- Claudeが学習できる明確なエラーメッセージを返す

In [60]:
def get_current_datetime(date_format="%Y-%m-%d %H:%M:%S"):
    """現在の日時を指定フォーマットで返す"""
    if not date_format:
        raise ValueError("date_format cannot be empty")
    return datetime.now().strftime(date_format)


# 動作確認
print(get_current_datetime())          # デフォルト: 2024-01-15 14:30:25
print(get_current_datetime("%H:%M"))   # 時刻のみ: 14:30
print(get_current_datetime("%Y年%m月%d日"))  # 日本語形式

2026-03-27 02:16:00
02:16
2026年03月27日


## 3. JSONスキーマの定義

Claudeにツールを認識させるためのスキーマを定義します。
スキーマはClaudeが「いつ・どう使うか」を理解するためのドキュメントです。

**3つの重要な要素：**
- `name` → ツール名
- `description` → 機能・使用タイミング・戻り値の説明
- `input_schema` → 引数の型・説明・必須かどうか

In [61]:
from anthropic.types import ToolParam

get_current_datetime_schema = ToolParam(
    name="get_current_datetime",
    description=(
        "指定されたフォーマットで現在の日時を返します。"
        "ユーザーが現在の時刻や日付を尋ねた場合、"
        "または日付計算のために現在の日時が必要な場合に使用してください。"
        "フォーマットされた日時の文字列を返します。"
    ),
    input_schema={
        "type": "object",
        "properties": {
            "date_format": {
                "type": "string",
                "description": (
                    "返される日時のフォーマットを指定する文字列。"
                    "Pythonのstrftimeフォーマットコードを使用します。"
                    "例: '%Y-%m-%d %H:%M:%S' は '2024-01-15 14:30:25' を返します。"
                ),
                "default": "%Y-%m-%d %H:%M:%S",
            }
        },
        "required": [],
    },
)

print("get_current_datetime_schema 定義完了")
print(get_current_datetime_schema)

get_current_datetime_schema 定義完了
{'name': 'get_current_datetime', 'description': '指定されたフォーマットで現在の日時を返します。ユーザーが現在の時刻や日付を尋ねた場合、または日付計算のために現在の日時が必要な場合に使用してください。フォーマットされた日時の文字列を返します。', 'input_schema': {'type': 'object', 'properties': {'date_format': {'type': 'string', 'description': "返される日時のフォーマットを指定する文字列。Pythonのstrftimeフォーマットコードを使用します。例: '%Y-%m-%d %H:%M:%S' は '2024-01-15 14:30:25' を返します。", 'default': '%Y-%m-%d %H:%M:%S'}}, 'required': []}}


## 4. ツール対応API呼び出しとマルチブロックメッセージの確認

`tools` パラメータにスキーマを渡すと、Claudeはテキストブロックとツール使用ブロックを含むマルチブロックメッセージを返します。

In [62]:
messages = []
messages.append({
    "role": "user",
    "content": "今の時刻を HH:MM:SS 形式で教えてください。",
})

response = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=messages,
    tools=[get_current_datetime_schema],
)

# レスポンス全体を確認
print("=== レスポンス全体 ===")
print(response)

print("\n=== content ブロック一覧 ===")
for i, block in enumerate(response.content):
    print(f"ブロック{i+1}: type={block.type}")
    print(block)

=== レスポンス全体 ===
Message(id='msg_017vuWpXaFPciMoaYwTzZFhp', container=None, content=[ToolUseBlock(id='toolu_01WhW7yYS4otiDYWbu8QXudJ', caller=DirectCaller(type='direct'), input={'date_format': '%H:%M:%S'}, name='get_current_datetime', type='tool_use')], model='claude-sonnet-4-5-20250929', role='assistant', stop_reason='tool_use', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=1246, output_tokens=62, server_tool_use=None, service_tier='standard'))

=== content ブロック一覧 ===
ブロック1: type=tool_use
ToolUseBlock(id='toolu_01WhW7yYS4otiDYWbu8QXudJ', caller=DirectCaller(type='direct'), input={'date_format': '%H:%M:%S'}, name='get_current_datetime', type='tool_use')


## 5. ToolUseブロックから情報を取り出す

`tool_use` ブロックには「どのツールをどんな引数で呼ぶか」が含まれています。

In [63]:
# tool_use ブロックを探して情報を取り出す
for block in response.content:
    if block.type == "tool_use":
        print(f"ツール名  : {block.name}")
        print(f"ツールID  : {block.id}")
        print(f"引数      : {block.input}")

# アシスタントメッセージを会話履歴に追加（マルチブロックごと保持）
messages.append({
    "role": "assistant",
    "content": response.content,  # テキストブロック＋ツール使用ブロックをそのまま保持
})

print("\n会話履歴に追加しました")

ツール名  : get_current_datetime
ツールID  : toolu_01WhW7yYS4otiDYWbu8QXudJ
引数      : {'date_format': '%H:%M:%S'}

会話履歴に追加しました


## 6. ツール実行結果をClaudeに返送する

`tool_result` ブロックを使ってツールの実行結果をClaudeに渡し、最終回答を得ます。

**返送の手順：**
1. `tool_use` ブロックのIDと同じ `tool_use_id` を指定する
2. `role: user` のメッセージとして `tool_result` ブロックを追加する
3. 再度APIを呼び出してClaudeの最終回答を取得する

In [64]:
# ツールディスパッチ関数：ツール名と引数からPython関数を呼び出す
def dispatch_tool(tool_name, tool_input):
    if tool_name == "get_current_datetime":
        return get_current_datetime(**tool_input)
    raise ValueError(f"未知のツール: {tool_name}")


# tool_use ブロックを実行して結果を返送する
tool_results = []
for block in response.content:
    if block.type == "tool_use":
        result = dispatch_tool(block.name, block.input)
        print(f"ツール実行: {block.name}({block.input}) → {result}")
        tool_results.append({
            "type": "tool_result",
            "tool_use_id": block.id,   # どのツール呼び出しへの返答かを紐付ける
            "content": result,
        })

# tool_result を user ロールのメッセージとして会話履歴に追加
messages.append({
    "role": "user",
    "content": tool_results,
})

print("\n会話履歴:")
for msg in messages:
    role = msg["role"]
    content = msg["content"]
    print(f"  [{role}] {content}")

ツール実行: get_current_datetime({'date_format': '%H:%M:%S'}) → 02:31:38

会話履歴:
  [user] 今の時刻を HH:MM:SS 形式で教えてください。
  [assistant] [ToolUseBlock(id='toolu_01WhW7yYS4otiDYWbu8QXudJ', caller=DirectCaller(type='direct'), input={'date_format': '%H:%M:%S'}, name='get_current_datetime', type='tool_use')]
  [user] [{'type': 'tool_result', 'tool_use_id': 'toolu_01WhW7yYS4otiDYWbu8QXudJ', 'content': '02:31:38'}]


In [65]:
# フォローアップリクエスト：ツール結果を含む会話履歴でClaudeに最終回答を求める
final_response = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=messages,
    tools=[get_current_datetime_schema],
)

print("=== Claudeの最終回答 ===")
for block in final_response.content:
    if block.type == "text":
        print(block.text)

print(f"\nstop_reason: {final_response.stop_reason}")  # end_turn になるはず

=== Claudeの最終回答 ===
現在の時刻は **02:31:38** です。

stop_reason: end_turn


## 7. 残り2つのツールを実装する

`add_time_to_datetime`：指定した日時に期間（日・時間・分）を加算して返します。
`set_reminder`：リマインダーを設定します（デモ用のインメモリ実装）。

In [66]:
def add_time_to_datetime(
    base_datetime: str,
    days: int = 0,
    hours: int = 0,
    minutes: int = 0,
    date_format: str = "%Y-%m-%d %H:%M:%S",
) -> str:
    """指定した日時に期間を加算して返す"""
    dt = datetime.strptime(base_datetime, date_format)
    dt += timedelta(days=days, hours=hours, minutes=minutes)
    return dt.strftime(date_format)


# デモ用リマインダーストレージ
reminders = []

def set_reminder(title: str, remind_at: str, date_format: str = "%Y-%m-%d %H:%M:%S") -> str:
    """リマインダーを設定してIDを返す（デモ用インメモリ実装）"""
    reminder_id = f"reminder_{len(reminders) + 1}"
    reminders.append({
        "id": reminder_id,
        "title": title,
        "remind_at": remind_at,
    })
    return f"リマインダーを設定しました。ID: {reminder_id}、タイトル: {title}、日時: {remind_at}"


# 動作確認
base = get_current_datetime()
print(f"現在時刻          : {base}")
print(f"2日後             : {add_time_to_datetime(base, days=2)}")
print(f"3時間30分後       : {add_time_to_datetime(base, hours=3, minutes=30)}")
print(set_reminder("会議の準備", add_time_to_datetime(base, hours=1)))

現在時刻          : 2026-03-27 02:33:40
2日後             : 2026-03-29 02:33:40
3時間30分後       : 2026-03-27 06:03:40
リマインダーを設定しました。ID: reminder_1、タイトル: 会議の準備、日時: 2026-03-27 03:33:40


## 8. 残り2つのスキーマを定義する

In [67]:
add_time_to_datetime_schema = ToolParam(
    name="add_time_to_datetime",
    description=(
        "指定した日時に期間（日・時間・分）を加算して返します。"
        "「3時間後」「2日後」など将来の日時を計算する場合に使用してください。"
        "フォーマットされた日時の文字列を返します。"
    ),
    input_schema={
        "type": "object",
        "properties": {
            "base_datetime": {
                "type": "string",
                "description": "加算の基準となる日時文字列。date_format に合わせた形式で指定します。",
            },
            "days": {
                "type": "integer",
                "description": "加算する日数。省略時は0。",
                "default": 0,
            },
            "hours": {
                "type": "integer",
                "description": "加算する時間数。省略時は0。",
                "default": 0,
            },
            "minutes": {
                "type": "integer",
                "description": "加算する分数。省略時は0。",
                "default": 0,
            },
            "date_format": {
                "type": "string",
                "description": "base_datetime および戻り値のフォーマット。デフォルトは '%Y-%m-%d %H:%M:%S'。",
                "default": "%Y-%m-%d %H:%M:%S",
            },
        },
        "required": ["base_datetime"],
    },
)

set_reminder_schema = ToolParam(
    name="set_reminder",
    description=(
        "指定した日時にリマインダーを設定します。"
        "ユーザーが「〜にリマインドして」「〜を忘れないようにして」と依頼した場合に使用してください。"
        "設定完了メッセージとリマインダーIDを返します。"
    ),
    input_schema={
        "type": "object",
        "properties": {
            "title": {
                "type": "string",
                "description": "リマインダーのタイトル（何をリマインドするか）。",
            },
            "remind_at": {
                "type": "string",
                "description": "リマインドする日時文字列。date_format に合わせた形式で指定します。",
            },
            "date_format": {
                "type": "string",
                "description": "remind_at のフォーマット。デフォルトは '%Y-%m-%d %H:%M:%S'。",
                "default": "%Y-%m-%d %H:%M:%S",
            },
        },
        "required": ["title", "remind_at"],
    },
)

all_tools = [get_current_datetime_schema, add_time_to_datetime_schema, set_reminder_schema]
print(f"登録ツール数: {len(all_tools)}")
for t in all_tools:
    print(f"  - {t['name']}")

登録ツール数: 3
  - get_current_datetime
  - add_time_to_datetime
  - set_reminder


## 9. 複数ツールを連鎖させる完全なフロー

「3時間後に会議をリマインドして」という依頼では、Claudeは以下の順番でツールを呼び出します：

1. `get_current_datetime` → 現在時刻を取得
2. `add_time_to_datetime` → 3時間後を計算
3. `set_reminder` → リマインダーを設定

ループでツール呼び出しが続く限り処理を繰り返し、`stop_reason == "end_turn"` になったら終了します。

In [68]:
def dispatch_tool(tool_name, tool_input):
    """ツール名と引数からPython関数を呼び出す"""
    if tool_name == "get_current_datetime":
        return get_current_datetime(**tool_input)
    if tool_name == "add_time_to_datetime":
        return add_time_to_datetime(**tool_input)
    if tool_name == "set_reminder":
        return set_reminder(**tool_input)
    raise ValueError(f"未知のツール: {tool_name}")


def run_with_tools(user_message: str) -> str:
    """Tool Useループ：end_turnになるまでツール呼び出しを繰り返す"""
    messages = [{"role": "user", "content": user_message}]
    step = 0

    while True:
        step += 1
        print(f"\n--- ステップ {step} ---")

        response = client.messages.create(
            model=model,
            max_tokens=1000,
            messages=messages,
            tools=all_tools,
        )
        print(f"stop_reason: {response.stop_reason}")

        # アシスタントの応答を会話履歴に追加
        messages.append({"role": "assistant", "content": response.content})

        # ツール呼び出しがなければ終了
        if response.stop_reason == "end_turn":
            for block in response.content:
                if block.type == "text":
                    return block.text

        # tool_use ブロックをすべて実行して結果を返送
        tool_results = []
        for block in response.content:
            if block.type == "tool_use":
                print(f"  ツール呼び出し: {block.name}({block.input})")
                result = dispatch_tool(block.name, block.input)
                print(f"  結果: {result}")
                tool_results.append({
                    "type": "tool_result",
                    "tool_use_id": block.id,
                    "content": result,
                })

        messages.append({"role": "user", "content": tool_results})


# 実行
reminders = []  # リマインダーをリセット
answer = run_with_tools("3時間後に『プロジェクトの進捗確認』をリマインドしてください。")
print(f"\n=== Claudeの最終回答 ===\n{answer}")
print(f"\n登録済みリマインダー: {reminders}")


--- ステップ 1 ---
stop_reason: tool_use
  ツール呼び出し: get_current_datetime({})
  結果: 2026-03-27 02:35:32

--- ステップ 2 ---
stop_reason: tool_use
  ツール呼び出し: add_time_to_datetime({'base_datetime': '2026-03-27 02:35:32', 'hours': 3})
  結果: 2026-03-27 05:35:32

--- ステップ 3 ---
stop_reason: tool_use
  ツール呼び出し: set_reminder({'title': 'プロジェクトの進捗確認', 'remind_at': '2026-03-27 05:35:32'})
  結果: リマインダーを設定しました。ID: reminder_1、タイトル: プロジェクトの進捗確認、日時: 2026-03-27 05:35:32

--- ステップ 4 ---
stop_reason: end_turn

=== Claudeの最終回答 ===
3時間後の**2026年3月27日 5:35**に『プロジェクトの進捗確認』のリマインダーを設定しました！

登録済みリマインダー: [{'id': 'reminder_1', 'title': 'プロジェクトの進捗確認', 'remind_at': '2026-03-27 05:35:32'}]


## 10. ヘルパー関数のリファクタリング

会話ループを汎用的に扱えるよう、ヘルパー関数を整備します。

- `add_user_message` → 文字列・ブロックリスト・Messageオブジェクトを統一処理
- `chat` → toolsパラメータ対応・Messageオブジェクトをそのまま返す
- `text_from_message` → Messageからテキストだけを抽出
- `run_tools` → tool_useブロックを実行してtool_resultブロックを返す

In [84]:
from anthropic.types import Message


def add_user_message(messages: list, message) -> None:
    """userメッセージを会話履歴に追加する。文字列・ブロックリスト・Messageオブジェクトに対応"""
    messages.append({
        "role": "user",
        "content": message.content if isinstance(message, Message) else message,
    })


def add_assistant_message(messages: list, message) -> None:
    """assistantメッセージを会話履歴に追加する。文字列・Messageオブジェクトに対応"""
    messages.append({
        "role": "assistant",
        "content": message.content if isinstance(message, Message) else message,
    })


def chat(messages: list, system=None, temperature=1.0, stop_sequences=[], tools=None, max_tokens=4096) -> Message:
    """APIを呼び出してMessageオブジェクトをそのまま返す"""
    params = {
        "model": model,
        "max_tokens": max_tokens,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }
    if tools:
        params["tools"] = tools
    if system:
        params["system"] = system
    return client.messages.create(**params)


def text_from_message(message: Message) -> str:
    """Messageオブジェクトからテキストブロックだけを結合して返す"""
    return "\n".join(block.text for block in message.content if block.type == "text")


def run_tools(response: Message) -> list:
    """tool_useブロックをすべて実行してtool_resultブロックのリストを返す（エラーハンドリング付き）"""
    tool_requests = [block for block in response.content if block.type == "tool_use"]
    tool_results = []

    for tool_request in tool_requests:
        print(f"  ツール呼び出し: {tool_request.name}({tool_request.input})")
        try:
            tool_output = dispatch_tool(tool_request.name, tool_request.input)
            print(f"  結果: {tool_output}")
            tool_results.append({
                "type": "tool_result",
                "tool_use_id": tool_request.id,
                "content": json.dumps(tool_output, ensure_ascii=False),
                "is_error": False,
            })
        except Exception as e:
            print(f"  エラー: {e}")
            tool_results.append({
                "type": "tool_result",
                "tool_use_id": tool_request.id,
                "content": f"Error: {e}",
                "is_error": True,
            })

    return tool_results


print("ヘルパー関数の定義完了")

ヘルパー関数の定義完了


## 11. 会話ループの実装と実行

ヘルパー関数を使って、マルチターンのツール呼び出しを自動処理する会話ループを実装します。

In [70]:
def run_conversation(user_input: str, tools: list) -> str:
    """ツール呼び出しが終わるまでループし、最終的なテキスト回答を返す"""
    messages = []
    add_user_message(messages, user_input)
    step = 0

    while True:
        step += 1
        print(f"\n--- ステップ {step} ---")

        response = chat(messages, tools=tools)
        print(f"stop_reason: {response.stop_reason}")

        # assistantの応答を会話履歴に追加
        add_assistant_message(messages, response)

        # ツール呼び出しがなければ終了
        if response.stop_reason == "end_turn":
            return text_from_message(response)

        # ツールを実行して結果を会話履歴に追加
        tool_results = run_tools(response)
        add_user_message(messages, tool_results)


# 実行：今日から103日後は何曜日か
answer = run_conversation("今日から103日後は何曜日ですか？", tools=all_tools)
print(f"\n=== 最終回答 ===\n{answer}")


--- ステップ 1 ---
stop_reason: tool_use
  ツール呼び出し: get_current_datetime({'date_format': '%Y-%m-%d %H:%M:%S'})
  結果: 2026-03-27 02:53:52

--- ステップ 2 ---
stop_reason: tool_use
  ツール呼び出し: add_time_to_datetime({'base_datetime': '2026-03-27 02:53:52', 'days': 103, 'date_format': '%Y-%m-%d %H:%M:%S'})
  結果: 2026-07-08 02:53:52

--- ステップ 3 ---
stop_reason: end_turn

=== 最終回答 ===
今日（2026年3月27日）から103日後は、**2026年7月8日（水曜日）**です。


## 12. エラーハンドリング付き run_tools のリファクタリング

ツール実行が失敗した場合も `is_error: True` のtool_resultブロックとしてClaudeに返します。
こうすることでClaudeがエラー内容を解釈して、ユーザーに適切なメッセージを返せます。

In [71]:
import json


def run_tools(response: Message) -> list:
    """tool_useブロックをすべて実行してtool_resultブロックのリストを返す（エラーハンドリング付き）"""
    tool_requests = [block for block in response.content if block.type == "tool_use"]
    tool_results = []

    for tool_request in tool_requests:
        print(f"  ツール呼び出し: {tool_request.name}({tool_request.input})")
        try:
            tool_output = dispatch_tool(tool_request.name, tool_request.input)
            print(f"  結果: {tool_output}")
            tool_results.append({
                "type": "tool_result",
                "tool_use_id": tool_request.id,
                "content": json.dumps(tool_output, ensure_ascii=False),
                "is_error": False,
            })
        except Exception as e:
            print(f"  エラー: {e}")
            tool_results.append({
                "type": "tool_result",
                "tool_use_id": tool_request.id,
                "content": f"Error: {e}",
                "is_error": True,
            })

    return tool_results


print("run_tools（エラーハンドリング付き）の定義完了")

run_tools（エラーハンドリング付き）の定義完了


In [72]:
# 正常系：今日から103日後は何曜日か
print("=== 正常系 ===")
answer = run_conversation("今日から103日後は何曜日ですか？", tools=all_tools)
print(f"\n最終回答: {answer}")

# エラー系：存在しないフォーマットを渡してエラーを発生させる
print("\n\n=== エラー系（不正な日時文字列） ===")
answer = run_conversation(
    "『abc』という日時の3日後を教えてください。",
    tools=all_tools,
)
print(f"\n最終回答: {answer}")

=== 正常系 ===

--- ステップ 1 ---
stop_reason: tool_use
  ツール呼び出し: get_current_datetime({'date_format': '%Y-%m-%d %H:%M:%S'})
  結果: 2026-03-27 02:56:13

--- ステップ 2 ---
stop_reason: tool_use
  ツール呼び出し: add_time_to_datetime({'base_datetime': '2026-03-27 02:56:13', 'days': 103, 'date_format': '%Y-%m-%d %H:%M:%S'})
  結果: 2026-07-08 02:56:13

--- ステップ 3 ---
stop_reason: end_turn

最終回答: 今日（2026年3月27日）から103日後は、**2026年7月8日（水曜日）**です。


=== エラー系（不正な日時文字列） ===

--- ステップ 1 ---
stop_reason: end_turn

最終回答: 申し訳ございません。「abc」は有効な日時形式ではないため、3日後の計算ができません。

日時を計算するには、以下のような形式で日時を指定していただく必要があります：

**例：**
- 2024-01-15 14:30:00
- 2024-01-15
- 2024/01/15 14:30

もしくは、「今日の3日後」や「現在時刻の3日後」を知りたい場合は、そのようにお伝えいただければ計算いたします。

どのような日時の3日後を知りたいですか？


## 13. 複数ツールを組み合わせたリクエストのテスト

「2050年1月1日から177日後の診察予約のリマインダーを設定してください。」

このリクエストではClaudeは現在時刻を知る必要がなく、以下の2ツールだけを使います：
1. `add_time_to_datetime` → 177日後の日付を計算
2. `set_reminder` → その日付にリマインダーを設定

In [73]:
reminders = []  # リマインダーをリセット

answer = run_conversation(
    "2050年1月1日から177日後の診察予約のリマインダーを設定してください。",
    tools=all_tools,
)

print(f"\n=== 最終回答 ===\n{answer}")
print(f"\n登録済みリマインダー: {reminders}")


--- ステップ 1 ---
stop_reason: tool_use
  ツール呼び出し: add_time_to_datetime({'base_datetime': '2050-01-01 00:00:00', 'days': 177})
  結果: 2050-06-27 00:00:00

--- ステップ 2 ---
stop_reason: tool_use
  ツール呼び出し: set_reminder({'title': '診察予約', 'remind_at': '2050-06-27 00:00:00'})
  結果: リマインダーを設定しました。ID: reminder_1、タイトル: 診察予約、日時: 2050-06-27 00:00:00

--- ステップ 3 ---
stop_reason: end_turn

=== 最終回答 ===
診察予約のリマインダーを設定しました！

**日時：2050年6月27日 00:00**（2050年1月1日から177日後）

リマインダーID: reminder_1 で登録されています。

登録済みリマインダー: [{'id': 'reminder_1', 'title': '診察予約', 'remind_at': '2050-06-27 00:00:00'}]


## 14. ストリーミング × Tool Use

ストリーミングを有効にすると、Claudeがツール引数を生成する様子をリアルタイムで確認できます。

**2種類のストリーミングモード：**
- **デフォルト（検証あり）** — APIがトップレベルのキー単位でJSON検証してからチャンクを送信。遅延あり・安全
- **fine_grained=True（検証なし）** — 生成されたチャンクを即座に送信。遅延なし・JSONが不完全な場合あり

In [74]:
def stream_with_tools(user_input: str, tools: list, fine_grained: bool = False) -> str:
    """ストリーミングでTool Useを処理する会話ループ"""
    messages = [{"role": "user", "content": user_input}]
    step = 0

    while True:
        step += 1
        print(f"\n--- ステップ {step} ---")

        stream_kwargs = {
            "model": model,
            "max_tokens": 1000,
            "messages": messages,
            "tools": tools,
        }
        if fine_grained:
            stream_kwargs["fine_grained"] = True

        with client.messages.stream(**stream_kwargs) as stream:
            # テキストブロックはリアルタイムで出力
            for chunk in stream:
                if chunk.type == "text":
                    print(chunk.text, end="", flush=True)

                # ツール引数のチャンクをリアルタイムで受信
                elif chunk.type == "input_json":
                    print(f"[JSON chunk] {repr(chunk.partial_json)}", end=" ", flush=True)

            response = stream.get_final_message()

        print(f"\nstop_reason: {response.stop_reason}")
        messages.append({"role": "assistant", "content": response.content})

        if response.stop_reason == "end_turn":
            return text_from_message(response)

        tool_results = run_tools(response)
        messages.append({"role": "user", "content": tool_results})


# デフォルト（JSON検証あり）でストリーミング
print("=== デフォルトストリーミング ===")
answer = stream_with_tools("今から2日と3時間後にリマインドして。タイトルは『定期メンテナンス』", tools=all_tools)
print(f"\n最終回答: {answer}")

=== デフォルトストリーミング ===

--- ステップ 1 ---
[JSON chunk] '' 
stop_reason: tool_use
  ツール呼び出し: get_current_datetime({})
  結果: 2026-03-27 03:05:33

--- ステップ 2 ---
[JSON chunk] '' [JSON chunk] '{"base_' [JSON chunk] 'datetime": ' [JSON chunk] '"2026' [JSON chunk] '-03-27 03:0' [JSON chunk] '5:33"' [JSON chunk] ', "days": 2' [JSON chunk] ', ' [JSON chunk] '"hours": 3}' 
stop_reason: tool_use
  ツール呼び出し: add_time_to_datetime({'base_datetime': '2026-03-27 03:05:33', 'days': 2, 'hours': 3})
  結果: 2026-03-29 06:05:33

--- ステップ 3 ---
[JSON chunk] '' [JSON chunk] '{"title": "' [JSON chunk] '定期メ' [JSON chunk] 'ンテナンス' [JSON chunk] '"' [JSON chunk] ', "remi' [JSON chunk] 'nd_at": "' [JSON chunk] '2026' [JSON chunk] '-03-29 06:' [JSON chunk] '05:33"}' 
stop_reason: tool_use
  ツール呼び出し: set_reminder({'title': '定期メンテナンス', 'remind_at': '2026-03-29 06:05:33'})
  結果: リマインダーを設定しました。ID: reminder_2、タイトル: 定期メンテナンス、日時: 2026-03-29 06:05:33

--- ステップ 4 ---
リマインダーを設定しました！

**タイトル:** 定期メンテナンス  
**日時:** 2026年3月29日 06:05:33

### fine_grained ストリーミングについて

`fine_grained=True` はドキュメントで紹介されている機能ですが、現在のSDKバージョンでは `messages.stream()` の直接パラメータとしてはサポートされていません。

将来的にはベータヘッダー経由での提供が想定されています：

```python
client.messages.stream(
    ...,
    extra_headers={"anthropic-beta": "fine-grained-tool-streaming-YYYY-MM-DD"}
)
```

**デフォルトとfine_grainedの概念的な違い：**

| | デフォルト | fine_grained |
|---|---|---|
| チャンクの粒度 | キー単位でまとめて送信 | 1文字単位でリアルタイム |
| JSON検証 | あり（API側でバリデーション） | なし |
| 用途 | 一般的なアプリケーション | 入力中の進捗をリアルタイム表示したい場合 |

今回のデフォルトストリーミングで、`[JSON chunk]` がキー単位でまとまって届いているのが確認できたのは、このバッファリング動作によるものです。

## 15. テキストエディタツール

Claudeに組み込まれているビルトインツールです。スキーマ定義は不要で、スタブ（小さな識別情報）を渡すだけでClaudeが完全なスペックを自動展開します。

**ただし実装（ファイル操作の実際の処理）は自分で用意する必要があります。**

| コマンド | 内容 |
|---|---|
| `view` | ファイル・ディレクトリの内容を表示 |
| `create` | 新規ファイルを作成 |
| `str_replace` | ファイル内のテキストを置換 |
| `insert` | 指定行にテキストを挿入 |
| `undo_edit` | 直前の編集を取り消す |

In [82]:
import os

# スキーマスタブ：モデルに応じたバージョン文字列を返す
def get_text_editor_schema(model: str) -> dict:
    if model.startswith("claude-3-5-sonnet"):
        return {"type": "text_editor_20241022", "name": "str_replace_editor"}
    elif model.startswith("claude-3-7-sonnet"):
        return {"type": "text_editor_20250124", "name": "str_replace_editor"}
    else:
        # Claude 4.x 以降：nameが str_replace_based_edit_tool に変更
        return {"type": "text_editor_20250728", "name": "str_replace_based_edit_tool"}


# ファイル操作の実装
def run_text_editor(command: str, **kwargs) -> str:
    """テキストエディタツールのコマンドを実際に実行する"""

    if command == "view":
        path = kwargs["path"]
        if os.path.isdir(path):
            entries = os.listdir(path)
            return "\n".join(sorted(entries))
        with open(path, "r", encoding="utf-8") as f:
            lines = f.readlines()
        # 行範囲指定があれば絞り込む
        view_range = kwargs.get("view_range")
        if view_range:
            start, end = view_range[0] - 1, view_range[1]
            lines = lines[start:end]
        return "".join(f"{i+1}: {line}" for i, line in enumerate(lines))

    elif command == "create":
        path = kwargs["path"]
        content = kwargs.get("file_text", "")
        os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
        with open(path, "w", encoding="utf-8") as f:
            f.write(content)
        return f"ファイルを作成しました: {path}"

    elif command == "str_replace":
        path = kwargs["path"]
        old_str = kwargs["old_str"]
        new_str = kwargs.get("new_str", "")
        with open(path, "r", encoding="utf-8") as f:
            content = f.read()
        if old_str not in content:
            return f"エラー: 指定した文字列が見つかりませんでした。"
        new_content = content.replace(old_str, new_str, 1)
        with open(path, "w", encoding="utf-8") as f:
            f.write(new_content)
        return f"テキストを置換しました: {path}"

    elif command == "insert":
        path = kwargs["path"]
        insert_line = kwargs["insert_line"]
        new_str = kwargs["new_str"]
        with open(path, "r", encoding="utf-8") as f:
            lines = f.readlines()
        lines.insert(insert_line, new_str + "\n")
        with open(path, "w", encoding="utf-8") as f:
            f.writelines(lines)
        return f"{insert_line}行目にテキストを挿入しました: {path}"

    elif command == "undo_edit":
        return "エラー: undo_editはこのデモ実装では未サポートです。"

    return f"エラー: 未知のコマンド: {command}"


text_editor_schema = get_text_editor_schema(model)
print(f"使用スキーマ: {text_editor_schema}")

使用スキーマ: {'type': 'text_editor_20250728', 'name': 'str_replace_based_edit_tool'}


In [85]:
TEXT_EDITOR_TOOL_NAMES = {"str_replace_editor", "str_replace_based_edit_tool"}

def dispatch_tool_with_editor(tool_name: str, tool_input: dict) -> str:
    """通常ツール＋テキストエディタツールのディスパッチ"""
    if tool_name in TEXT_EDITOR_TOOL_NAMES:
        command = tool_input.pop("command")
        return run_text_editor(command, **tool_input)
    return dispatch_tool(tool_name, tool_input)


def run_conversation_with_editor(user_input: str, tools: list) -> str:
    """テキストエディタツール対応の会話ループ"""
    messages = [{"role": "user", "content": user_input}]
    step = 0

    while True:
        step += 1
        print(f"\n--- ステップ {step} ---")
        response = chat(messages, tools=tools)
        print(f"stop_reason: {response.stop_reason}")
        add_assistant_message(messages, response)

        if response.stop_reason == "end_turn":
            return text_from_message(response)

        tool_results = []
        for block in response.content:
            if block.type == "tool_use":
                print(f"  コマンド: {block.input.get('command')} / {block.input}")
                try:
                    result = dispatch_tool_with_editor(block.name, dict(block.input))
                    print(f"  結果: {result[:100]}...")
                    tool_results.append({
                        "type": "tool_result",
                        "tool_use_id": block.id,
                        "content": result,
                        "is_error": False,
                    })
                except Exception as e:
                    tool_results.append({
                        "type": "tool_result",
                        "tool_use_id": block.id,
                        "content": f"Error: {e}",
                        "is_error": True,
                    })

        add_user_message(messages, tool_results)


# テスト：main.pyを作成してtest.pyでテストを生成させる
answer = run_conversation_with_editor(
    "./main.py ファイルを開いて、円周率を小数点以下第5位まで計算する関数を記述してください。"
    "次に、./test.py ファイルを作成して、実装をテストしてください。",
    tools=[text_editor_schema],
)
print(f"\n=== 最終回答 ===\n{answer}")


--- ステップ 1 ---
stop_reason: tool_use
  コマンド: view / {'command': 'view', 'path': './'}
  結果: .claude
.env
.gitignore
.ipynb_checkpoints
.venv
001_prompt_evals.ipynb
002_tool_use.ipynb
dataset.j...

--- ステップ 2 ---
stop_reason: tool_use
  コマンド: view / {'command': 'view', 'path': './main.py'}
  結果: 1: """円周率を計算するモジュール
2: 
3: ライプニッツの公式を使用して円周率を近似計算します。
4: π/4 = 1 - 1/3 + 1/5 - 1/7 + 1/9 - ...
5: ""...

--- ステップ 3 ---
stop_reason: tool_use
  コマンド: create / {'command': 'create', 'path': './test.py', 'file_text': '"""円周率計算関数のテストモジュール\n\nmain.pyで実装された円周率計算関数をテストします。\n"""\n\nimport unittest\nimport math\nfrom main import calculate_pi, calculate_pi_machin\n\n\nclass TestPiCalculation(unittest.TestCase):\n    """円周率計算関数のテストクラス"""\n    \n    def test_calculate_pi_default_precision(self):\n        """デフォルトの精度（小数点以下5桁）でテスト"""\n        result = calculate_pi()\n        expected = 3.14159\n        self.assertEqual(result, expected)\n        print(f"✓ calculate_pi() = {result} (期待値: {expected})")\n    \

## 16. Web検索ツール

テキストエディタツールと同様のビルトインツールです。
スキーマを渡すだけでClaudeが検索処理を自動的に行います（実装不要）。

事前に https://console.anthropic.com/settings/privacy でWeb検索を有効化する必要があります。

In [86]:
web_search_schema = {
    "type": "web_search_20250305",
    "name": "web_search",
    "max_uses": 5,  # 1リクエスト内でClaudeが実行できる最大検索回数
}

response = client.messages.create(
    model=model,
    max_tokens=4096,
    tools=[web_search_schema],
    messages=[{"role": "user", "content": "2025年に話題になったAIモデルを教えてください。"}],
)

# レスポンスのブロック構造を確認する
print("=== ブロック一覧 ===")
for block in response.content:
    print(f"\ntype: {block.type}")
    print(block)

=== ブロック一覧 ===

type: server_tool_use
ServerToolUseBlock(id='srvtoolu_01PCd9LpNsb734vZ9MweFLH7', caller=DirectCaller(type='direct'), input={'query': '2025年 話題 AIモデル'}, name='web_search', type='server_tool_use')

type: web_search_tool_result
WebSearchToolResultBlock(caller=DirectCaller(type='direct'), content=[WebSearchResultBlock(encrypted_content='EtMgCioIDhgCIiQ2Yjk5ZTNkMy1mZjg1LTQ4ZDUtOTRlMC02ZjNkNGM2YjEyMjMSDJhjiMxWynop6U/omhoMfjCE19IDOCx0ob/wIjBp5fOJZSiqQLLN7Ivmm0a25s4Ntjm7irMfL9KjejPvLis93QKRVMYzsrvWmtyEfEQq1h9WAZlhHLplQZWTsSR+aWhQEvB/MWH1cbOnfsVaJdtfy8rSW6zAV44GizMOzwWOdZcXtlZq0oFDkMJxlo6zd/DKQ3RNHLaNQPLVZiE7hNzo7vN8qnmu5iVuwP4dAJ3s6vZRdFharZHhUBLBI5uEmqY5XTzziL6d8BDpNgXDStVYcQBQO7aUtG2nuf5+8+KQkgba98/Ze5CEGGKyJpq9GXYjuWTCMe0DwIEdBZ81vUPbmFA6dlNpD93LTtpI/N9hxxhPviNNiLzB+bEM8OnjAZllxOiVvCSJ/ZKq8cnRkUzKOwTv5UYRGOVivIunjaubohbKF59cl3H5zK57tTozyrEjLVeYChDsGcGEtbUWizvhd04go/RZyx9IH07vnt5fPla3sJkWe6c3A+b3aNNadCN7/U5rLJDAe/xX0EEeifE96JxBmrCOWRFF+3nA+xOS2W+fMxcDyj2vPSaqlvFzr3er7beiVa/Da

In [87]:
# 各ブロックタイプを解析して整形出力する
print("=== 検索クエリ ===")
for block in response.content:
    if block.type == "server_tool_use":
        print(f"  検索クエリ: {block.input.get('query')}")

print("\n=== テキスト回答 ===")
for block in response.content:
    if block.type == "text":
        print(block.text)

print("\n=== 引用元 ===")
for block in response.content:
    if block.type == "web_search_tool_result":
        for result in block.content:
            if hasattr(result, "url"):
                print(f"  - {result.title}")
                print(f"    {result.url}")

=== 検索クエリ ===
  検索クエリ: 2025年 話題 AIモデル

=== テキスト回答 ===
2025年に話題になった主要なAIモデルをご紹介します。

## OpenAIのモデル


**GPT-4.5（コードネーム：Orion）**が2025年2月28日に正式発表され、感情的知能（EQ）の向上、ハルシネーションの減少、創造性と直感力の強化が実現されました
。
3月6日からはPlus（月額20ドル）ユーザーでも利用可能になりました
。


**GPT-5**は2025年8月8日に発表され、「これまでで最も賢く、有用なモデル」と位置づけられ、主要なベンチマークで最高水準を達成しました
。

## 中国発のAIモデル


**DeepSeek-R1**が1月にリリースされ、OpenAIのモデルに匹敵する推論能力を持ちながらトレーニングコストが桁違いに安く、NVIDIAの株価を一時的に揺るがす事態となりました
。
**DeepSeek V3.1**も8月19日にリリースされ、特に推論能力が強化され、より長い文章を処理できるようになりました
。


中国アリババは8月19日に画像編集に特化した**Qwen-Image-Edit**を発表しました
。

## Googleのモデル


**Gemini 2.5 Flash Image**が8月26日に発表され、被写体の特徴やスタイルを保ったまま自然な画像編集ができるのが最大の特徴です
。
Google Geminiが2025年で最も飛躍したとする評価もあります
。

## その他の注目モデル


Anthropicは「**Claude Opus 4.5**」を投入し、コーディングや複雑な操作において「仕事で使うならClaude」という地位を固めました
。

2025年は、
各企業が「こっちの方が賢い」「いや、あっちの方が速い」とニュースが飛び交い、情報が多すぎて追い切れないほど激しい競争の一年
となりました。

=== 引用元 ===
  - 【2026】AI最新情報15選！ChatGPTの新技術・面白いAIニュース・最新AIを解説 | DX/AI研究所
    https://ai-kenkyujo.com/news/ai-saishin/
  - 生成AIモデル乱立！2025年版 最新モデル総まとめ - G